## Who's Buying and Selling Their Own Stock?

This code does a local scrape of the 100 most recent Form 4 filings from SEC EDGAR, before I set up the updater.

Form 4s are what company insiders — directors, officers, and big shareholders — have to file when they buy or sell shares in their own company. So this is basically insider trading data.

I first get all the fields from the main list page, and then I follow each filing to its Form 4 XML to get the transaction details and the footnote text.

This is the page I am scraping:
* [SEC EDGAR — latest Form 4 filings](https://www.sec.gov/cgi-bin/browse-edgar?company=&CIK=&type=4&owner=include&count=100&action=getcurrent)

In [1]:
#FUNCTIONS FOR EXTRACTING DATA POINTS
#Layer 1: pull EVERY field out of a getcurrent table row-pair into a dict
#Layer 2: "slice" deeper -> follow the filing index -> parse the Form 4 XML (text + data)

import re
from bs4 import BeautifulSoup

BASE = "https://www.sec.gov"


def extract_row_fields(name_cell, data_row):
    #LAYER 1: everything visible in the getcurrent contents table.
    #Each filing occupies TWO <tr>s: a 'name row' (entity + CIK + role) and a
    #'data row' (form, format links, description, dates, file/film no).
    fields = {}

    #name row: e.g. "Richard Christian M (0001645967) (Reporting)"
    a = name_cell.find('a')
    raw_name = a.get_text(strip=True)
    fields["entity_link"] = BASE + a['href']
    m = re.match(r'^(.*?)\s*\((\d{10})\)\s*\((\w+)\)\s*$', raw_name)
    if m:
        fields["entity_name"] = m.group(1).strip()
        fields["entity_cik"]  = m.group(2)
        fields["entity_role"] = m.group(3)          # Reporting or Issuer
    else:
        fields["entity_name"] = raw_name            # fallback if pattern differs

    #data row: iterate the cells (issuer rows have 5, reporting rows 6)
    tds = data_row.find_all('td', recursive=False)
    fields["form_type"] = tds[0].get_text(strip=True)

    #format links: [html] index + [text] complete submission
    for link in tds[1].find_all('a'):
        if 'html' in link.text: fields["html_index_link"] = BASE + link['href']
        if 'text' in link.text: fields["txt_link"]        = BASE + link['href']

    #description cell also carries "Accession Number / Act / Size"
    desc_cell = tds[2]
    fields["description"] = desc_cell.contents[0].strip() if desc_cell.contents else ""
    meta_text = desc_cell.get_text(" ", strip=True)
    acc  = re.search(r'Accession Number:\s*(\S+)', meta_text)
    act  = re.search(r'Act:\s*(\S+)', meta_text)
    size = re.search(r'Size:\s*([\d\.]+\s*\w+)', meta_text)
    if acc:  fields["accession_number"] = acc.group(1)
    if act:  fields["act"]  = act.group(1)
    if size: fields["size"] = size.group(1)

    #accepted date + time (two lines separated by <br>)
    acc_parts = list(tds[3].stripped_strings)
    if acc_parts:
        fields["accepted_date"] = acc_parts[0]
        if len(acc_parts) > 1: fields["accepted_time"] = acc_parts[1]

    fields["filing_date"] = tds[4].get_text(strip=True)

    #file/film number only present on some rows -> guard it
    if len(tds) > 5:
        parts = list(tds[5].stripped_strings)
        if parts:              fields["file_number"] = parts[0]
        if len(parts) > 1:     fields["film_number"] = parts[1]

    return fields


def _gv(node, tag):
    #helper: value of a child tag's <value> (or its own text), else None
    el = node.find(tag)
    if not el: return None
    v = el.find('value')
    txt = (v.get_text(strip=True) if v else el.get_text(strip=True))
    return txt or None


def _parse_txn(t):
    #one transaction (works for both derivative + non-derivative)
    d = {
        "security_title":        _gv(t, 'securityTitle'),
        "transaction_date":      _gv(t, 'transactionDate'),
        "shares_owned_following":_gv(t, 'sharesOwnedFollowingTransaction'),
        "direct_or_indirect":    _gv(t, 'directOrIndirectOwnership'),
    }
    coding = t.find('transactionCoding')
    if coding and coding.find('transactionCode'):
        d["transaction_code"] = coding.find('transactionCode').get_text(strip=True)
    amts = t.find('transactionAmounts')
    if amts:
        d["shares"]            = _gv(amts, 'transactionShares')
        d["price_per_share"]   = _gv(amts, 'transactionPricePerShare')
        d["acquired_disposed"] = _gv(amts, 'transactionAcquiredDisposedCode')
    if t.find('conversionOrExercisePrice'):
        d["conversion_exercise_price"] = _gv(t, 'conversionOrExercisePrice')
    if t.find('underlyingSecurity'):
        d["underlying_title"]  = _gv(t, 'underlyingSecurityTitle')
        d["underlying_shares"] = _gv(t, 'underlyingSecurityShares')
    return {k: v for k, v in d.items() if v is not None}   # drop empties -> flexible schema


def extract_form4_data(index_soup, xml_soup):
    #LAYER 2: deep fields sliced from the filing index + the Form 4 XML.
    #This is where the actual TEXT (footnotes) and per-transaction data live.
    f = {}

    #list of all documents attached to the filing (from the index page)
    f["documents"] = []
    table = index_soup.find('table', class_='tableFile')
    for r in (table.find_all('tr') if table else []):
        cells = [c.get_text(strip=True) for c in r.find_all(['th', 'td'])]
        if not cells or cells[0] == 'Seq':
            continue
        f["documents"].append({
            "cells": cells,
            "links": [BASE + a['href'] for a in r.find_all('a')],
        })

    if xml_soup is None:
        return f    # some filings have no parseable XML -> still return index data

    #header-level fields
    for tag, key in [('schemaVersion','schema_version'), ('documentType','document_type'),
                     ('periodOfReport','period_of_report')]:
        el = xml_soup.find(tag)
        if el and el.text.strip(): f[key] = el.text.strip()

    iss = xml_soup.find('issuer')
    if iss:
        if iss.find('issuerCik'):  f["issuer_cik"]  = iss.find('issuerCik').text.strip()
        if iss.find('issuerName'): f["issuer_name"] = iss.find('issuerName').text.strip()
        sym = iss.find('issuerTradingSymbol')
        if sym and sym.text.strip(): f["issuer_trading_symbol"] = sym.text.strip()

    ro = xml_soup.find('reportingOwner')
    if ro:
        rid = ro.find('reportingOwnerId')
        if rid:
            if rid.find('rptOwnerCik'):  f["reporting_owner_cik"]  = rid.find('rptOwnerCik').text.strip()
            if rid.find('rptOwnerName'): f["reporting_owner_name"] = rid.find('rptOwnerName').text.strip()
        addr = ro.find('reportingOwnerAddress')
        if addr:
            f["reporting_owner_address"] = {c.name: c.get_text(strip=True)
                                            for c in addr.find_all(recursive=False)
                                            if c.get_text(strip=True)}
        rel = ro.find('reportingOwnerRelationship')
        if rel:
            f["relationship"] = {c.name: c.get_text(strip=True)
                                 for c in rel.find_all(recursive=False)
                                 if c.get_text(strip=True)}

    #transactions (either table may be empty)
    nd = [_parse_txn(t) for t in xml_soup.find_all('nonDerivativeTransaction')]
    dv = [_parse_txn(t) for t in xml_soup.find_all('derivativeTransaction')]
    if nd: f["non_derivative_transactions"] = nd
    if dv: f["derivative_transactions"]     = dv

    #the actual TEXT of the filing: footnotes
    fns = [fn.get_text(strip=True) for fn in xml_soup.find_all('footnote')]
    if fns: f["footnotes"] = fns

    sig = xml_soup.find('ownerSignature')
    if sig:
        if sig.find('signatureName'): f["signature_name"] = sig.find('signatureName').text.strip()
        if sig.find('signatureDate'): f["signature_date"] = sig.find('signatureDate').text.strip()

    return f

In [2]:
#FETCH + PARSE ONE FILING'S DEEP DATA (index page -> Form 4 XML)
import requests
import time

def get_form4_details(index_url, head):
    #Follow the filing's HTML index, find the raw Form 4 XML, and parse both.
    print(index_url)
    index_html = requests.get(index_url, headers=head, timeout=120).content
    index_soup = BeautifulSoup(index_html, "html.parser")

    #locate the raw form-4 xml (skip the xslF345 rendered version)
    xml_url = None
    table = index_soup.find('table', class_='tableFile')
    for a in (table.find_all('a') if table else []):
        href = a['href']
        if href.endswith('.xml') and 'xslF345' not in href:
            xml_url = BASE + href
            break

    xml_soup = None
    if xml_url:
        time.sleep(0.3)                      # be polite to SEC (<10 req/sec)
        xml_bytes = requests.get(xml_url, headers=head, timeout=120).content
        xml_soup = BeautifulSoup(xml_bytes, "xml")

    details = extract_form4_data(index_soup, xml_soup)
    if xml_url: details["xml_url"] = xml_url
    return details

In [3]:
#SCRAPING LOGIC

#IMPORTS
import os, json, hashlib, datetime, traceback, requests, time, re
from bs4 import BeautifulSoup

#STEP ONE: check for data history
DATA_FILE = "data/sec_form4.json"
TODAY_STR = datetime.date.today().isoformat()
os.makedirs("data/changelogs", exist_ok=True)
os.makedirs("data/error_logs", exist_ok=True)
os.makedirs(os.path.dirname(DATA_FILE), exist_ok=True)

master_list = []
if os.path.exists(DATA_FILE):
    print("Found existing dataset. Loading history...")
    with open(DATA_FILE) as fh:
        master_list = json.load(fh)
    old_data_map = {item['url']: item for item in master_list}   # key by index url
else:
    print("No existing dataset found. Initializing a baseline run...")
    old_data_map = {}

#STEP TWO: change log + error log
changelog = {"date": TODAY_STR, "additions": [], "deletions": [], "modifications": []}
error_log = {"date": TODAY_STR, "errors": []}

#STEP THREE: scrape the getcurrent contents page
todays_filings = []

#SEC requires a descriptive UA with real contact info
HEAD = {'User-Agent': 'Dimuthu Attanayake dca2140@columbia.edu'}

CONTENTS_URL = ("https://www.sec.gov/cgi-bin/browse-edgar?"
                "company=&CIK=&type=4&owner=include&count=100&action=getcurrent")
raw_html = requests.get(CONTENTS_URL, headers=HEAD, timeout=120).content
contents_doc = BeautifulSoup(raw_html, "html.parser")

#find the filings table (the one whose header has "Formats" + "Filing Date")
main_table = None
for t in contents_doc.find_all('table'):
    ths = [th.get_text(strip=True) for th in t.find_all('th')]
    if 'Formats' in ths and 'Filing Date' in ths:
        main_table = t
        break

#LAYER 1: pair each name row with its data row -> full dict of table fields
row_entries = []
current_name = None
for r in main_table.find_all('tr'):
    if r.find('th'):                       # skip header
        continue
    name_cell = r.find('td', bgcolor="#E6E6E6")
    if name_cell:                          # this is a name row
        current_name = name_cell
        continue
    if current_name is not None:           # this is the matching data row
        row_entries.append(extract_row_fields(current_name, r))
        current_name = None

current_urls = [e["html_index_link"] for e in row_entries if "html_index_link" in e]

#check for deleted items (fell off the latest-100 feed)
#getcurrent is a rolling feed, so "deletion" just means it aged out of the
#latest 100 -- we log it but DON'T drop it from the master list.
for url, old_item in old_data_map.items():
    if url not in current_urls:
        changelog["deletions"].append({"url": url,
                                        "entity_name": old_item.get("entity_name"),
                                        "accession_number": old_item.get("accession_number")})

#STEP FOUR: check / scrape each filing
for entry in row_entries:                  # all 100; slice [:5] to test
    url = entry.get("html_index_link")
    yesterdays_item = old_data_map.get(url)
    time.sleep(0.3)                        # polite pacing
    try:
        #change-detection hash: filings are immutable, so hash the stable identifiers
        hash_string = (entry.get("accession_number","") + entry.get("accepted_time","")
                       + entry.get("entity_cik","")).lower()
        hash_id = hashlib.md5(hash_string.encode('utf-8')).hexdigest()

        if url not in old_data_map:                       # NEW filing -> get everything
            print("new entry")
            entry["content_hash"] = hash_id
            entry["url"] = url
            entry["entry_id"] = entry.get("entity_cik","") + "_" + entry.get("accession_number","")
            entry.update(get_form4_details(url, HEAD))     # LAYER 2 deep slice
            todays_filings.append(entry)
            changelog["additions"].append({"entry_id": entry["entry_id"], "url": url,
                                           "entity_name": entry.get("entity_name")})
        else:
            yesterdays_hash = yesterdays_item['content_hash']
            if yesterdays_hash == hash_id:                 # NO change
                print("they match!")
                todays_filings.append(yesterdays_item)
            else:                                          # CHANGED
                print("no match")
                entry["content_hash"] = hash_id
                entry["url"] = url
                entry["entry_id"] = entry.get("entity_cik","") + "_" + entry.get("accession_number","")
                entry.update(get_form4_details(url, HEAD))
                todays_filings.append(entry)
                meaningful_changes = {k: {"from": yesterdays_item.get(k), "to": v}
                                      for k, v in entry.items() if yesterdays_item.get(k) != v}
                if meaningful_changes:
                    changelog["modifications"].append({"entry_id": entry["entry_id"],
                                                       "changes": meaningful_changes})
    except Exception as e:
        print(f"❌ Error scraping {url}: {str(e)}")
        if yesterdays_item:                    # keep yesterday's copy on failure
            todays_filings.append(yesterdays_item)
        error_log["errors"].append({
            "url": url,
            "entity_name": entry.get("entity_name"),
            "error_type": type(e).__name__,
            "message": str(e),
            "traceback": traceback.format_exc().splitlines()[-3:],
        })

#merge today's filings into the accumulating master list (grows to 500-1000+)
for item in todays_filings:
    old_data_map[item["url"]] = item
updated_list = list(old_data_map.values())

#STEP FIVE: write files
with open(DATA_FILE, 'w') as fh:
    sorted_data = sorted(updated_list, key=lambda x: x.get('entry_id', x.get('url','')))
    json.dump(sorted_data, fh, indent=2)
if changelog["additions"] or changelog["deletions"] or changelog["modifications"]:
    with open(f"data/changelogs/{TODAY_STR}.json", 'w') as fh:
        json.dump(changelog, fh, indent=2)
if error_log["errors"]:
    with open(f"data/error_logs/{TODAY_STR}.json", 'w') as fh:
        json.dump(error_log, fh, indent=2)

print(f"Scrape done! {len(updated_list)} total filings in dataset.")

Found existing dataset. Loading history...


they match!


they match!


they match!


they match!


...

they match!


they match!


they match!


they match!


they match!
Scrape done! 100 total filings in dataset.
